## 課題
今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) をソフトマックス回帰によって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値 (Target)
Accuracy: 80%

### ルール
- 訓練データは`x_train`， `y_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train、y_train`以外の学習データは使わないでください．**
- **隠れ層のない単層のモデルとして実装してください.**
- **ソフトマックス回帰のアルゴリズム部分の実装はnumpyのみで行ってください** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

In [2]:
# Library Import 
import pandas as pd
import numpy as np
import os
import sys

In [3]:
# モジュールを固定する。
# sys.module に None を入れておくと、以降 import tesorflow が ImportError になる。
# アルゴリズムは numpy のみというルールを抜けないようにしている。
sys.modules['tensorflow'] = None

In [4]:
# データの格納場所の指定
work_dir = '/Users/ryosuke/ai-engneering/data/jdla-e/lecture02/'

In [5]:
# ①データの読み込み
# .npy は Numpy 配列を保存したファイル。読み込むと ndarray が返る。
x_test = np.load(work_dir + 'x_test.npy')   # テスト画像（予測対象なので、正解ラベルは渡されない。）
x_train = np.load(work_dir + 'x_train.npy') # 訓練画像
y_train = np.load(work_dir + 'y_train.npy') # 訓練ラベル（0~9の整数)    

In [8]:
# ②データの前処理
# 画像: (N, 28, 28) -> (N, 784) に平坦化し、画素値 0~255 を 0~1 に正規化を行う。

x_train = x_train.reshape(-1, 784).astype('float32') / 255
x_test = x_test.reshape(-1, 784).astype('float32') / 255

'''
もともとMNISTの画像は以下のような形になっている。
1枚の画像 = 28ピクセル * 28ピクセル
つまり、1枚の画像は二次元の表です。
[
  [0, 0, 12, ...],
  [0, 35, 200, ...],
  ...
]
これをそのまま、計算すると大変になるので、1枚の画像を784(=28*28)のピクセルが並んだ１本のベクトルとして扱う。
そのため、(N, 784)に平坦化している。　※Nは画像の枚数を指す。
そして画像の各ピクセルは、通常 0〜255 の整数です。
0   = 黒
255 = 白
このままだと数値が大きいので、機械学習ではよく 0〜1 に変換します。
なので、/255 をしてあげることで、0~255の画素値が 0~1の間に収めることができるようになる。
'''

'\nもともとMNISTの画像は以下のような形になっている。\n1枚の画像 = 28ピクセル * 28ピクセル\nつまり、1枚の画像は二次元の表です。\n[\n  [0, 0, 12, ...],\n  [0, 35, 200, ...],\n  ...\n]\nこれをそのまま、計算すると大変になるので、1枚の画像を784(=28*28)のピクセルが並んだ１本のベクトルとして扱う。\nそのため、(N, 784)に平坦化している。\u3000※Nは画像の枚数を指す。\nそして画像の各ピクセルは、通常 0〜255 の整数です。\n0   = 黒\n255 = 白\nこのままだと数値が大きいので、機械学習ではよく 0〜1 に変換します。\nなので、/255 をしてあげることで、0~255の画素値が 0~1の間に収めることができるようになる。\n'

In [9]:
# ③ラベルをOne hot にする。
y_train = np.eye(10)[y_train.astype('int32')]

'''
np.eye(10)は、10×10の単位行列を作る。
[
 [1,0,0,0,0,0,0,0,0,0],
 [0,1,0,0,0,0,0,0,0,0],
 [0,0,1,0,0,0,0,0,0,0],
 ...
 [0,0,0,0,0,0,0,0,0,1]
]

上記に対して、正解ラベルが'3'だった場合は、
np.eye(10)[3]
によって、[0,0,1,0,0,0,0,0,0,0]を取得することができる。
最終的にソフトマックス関数のと正解ラベルが以下のようになった場合に、それぞれのクロスエントロピー誤差を計算する。
正解：[0,0,0,1,0,0,0,0,0,0]
予測：[0.01,0.02,0.03,0.80,0.01,0.02,0.01,0.05,0.03,0.02]
'''

"\nnp.eye(10)は、10×10の単位行列を作る。\n[\n [1,0,0,0,0,0,0,0,0,0],\n [0,1,0,0,0,0,0,0,0,0],\n [0,0,1,0,0,0,0,0,0,0],\n ...\n [0,0,0,0,0,0,0,0,0,1]\n]\n\n上記に対して、正解ラベルが'3'だった場合は、\nnp.eye(10)[3]\nによって、[0,0,1,0,0,0,0,0,0,0]を取得することができる。\n最終的にソフトマックス関数のと正解ラベルが以下のようになった場合に、それぞれのクロスエントロピー誤差を計算する。\n正解：[0,0,0,1,0,0,0,0,0,0]\n予測：[0.01,0.02,0.03,0.80,0.01,0.02,0.01,0.05,0.03,0.02]\n"

In [10]:
# ===== ③パラメータを初期化する。 =====


In [11]:
# ===== ④ソフトマックス関数を定義する。 =====


In [12]:
# ===== ⑤クロスエントロピー誤差を定義する。 =====


In [13]:
# ===== ⑥学習する。 =====

In [14]:
# ===== ⑦テストデータを予測する。 =====
